# Gold — Analyse comptable
Pipeline Olist : indicateurs financiers à partir de la zone silver.

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as _sum, avg, count, round as _round,
    countDistinct, date_format, max as _max,
    collect_set, concat_ws, create_map, lit, coalesce
)
from itertools import chain

spark = SparkSession.builder \
    .appName("olist-gold-account") \
    .getOrCreate()



## Chargement des tables silver

In [ ]:
df_orders = spark.read.parquet("../data/silver/orders/")
df_customers = spark.read.parquet("../data/silver/customers/")
df_items = spark.read.parquet("../data/silver/order_items/")
df_payments = spark.read.parquet("../data/silver/payments/")
df_products = spark.read.parquet("../data/silver/products/")
df_sellers = spark.read.parquet("../data/silver/sellers/")
df_pcnt = spark.read.parquet("../data/silver/product_category_name_translation/")

## Segmentation des commandes par statut
Trois segments : revenu reconnu, en transit, exclu.

In [ ]:
df_orders.groupBy("order_status").count().show()

In [ ]:
# Trois segments selon le statut de la commande
df_orders_delivered = df_orders.filter(col("order_status") == "delivered")

df_orders_in_transit = df_orders.filter(
    col("order_status").isin(["created", "approved", "processing", "invoiced", "shipped"])
)

df_orders_excluded = df_orders.filter(
    col("order_status").isin(["canceled", "unavailable"])
)

## Chiffre d'affaires par segment
Jointure order_items + orders

In [ ]:
df_gold_revenue_delivered = df_items.join(
    df_orders_delivered.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

df_gold_revenue_in_transit = df_items.join(
    df_orders_in_transit.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

df_gold_revenue_excluded = df_items.join(
    df_orders_excluded.select("order_id", "customer_id", "order_purchase_timestamp"),
    "order_id", "inner"
)

In [ ]:
# Fonction pour calculer le chiffre d'affaires d'un segment donné
def calculate_revenue(df, name):
    result = df.agg(
        _round(_sum("price"), 2).alias("ca_produits"),
        _round(_sum("freight_value"), 2).alias("total_frais_livraison"),
        _round(_sum("price") + _sum("freight_value"), 2).alias("ca_total")
    )
    print(f"=== {name} ===")
    result.show()
    return result

ca_delivered = calculate_revenue(df_gold_revenue_delivered, "Revenu reconnu (delivered)")
ca_in_transit = calculate_revenue(df_gold_revenue_in_transit, "Revenu en transit (payé, non livré)")
ca_excluded = calculate_revenue(df_gold_revenue_excluded, "Exclu (annulé/indisponible)")

## Chiffre d'affaires mensuel
Évolution du revenu reconnu dans le temps.

In [ ]:
df_monthly_revenue = df_gold_revenue_delivered.withColumn(
    "month", date_format(col("order_purchase_timestamp"), "yyyy-MM")
).groupBy("month").agg(
    _round(_sum("price"), 2).alias("ca_produits"),
    _round(_sum("freight_value"), 2).alias("total_frais_livraison"),
    _round(_sum("price") + _sum("freight_value"), 2).alias("ca_total")
).orderBy("month")

df_monthly_revenue.show(30)

## Validation — paiements par commande
Vérification de la cardinalité avant toute jointure avec payments.

In [ ]:
# Combien de paiements par commande ?
payments_per_order = df_payments.groupBy("order_id").agg(count("*").alias("nb_payments"))
payments_per_order.groupBy("nb_payments").count().orderBy("nb_payments").show()

In [ ]:
# Inspection d'un cas extrême (commande avec le plus de paiements)
example_order_id = df_payments.groupBy("order_id").count().orderBy(col("count").desc()).first()["order_id"]
df_payments.filter(col("order_id") == example_order_id).orderBy("payment_sequential").show(30)

## Répartition des paiements par type
Valeur totale, nombre de transactions et part de chaque type.

In [ ]:
total_payments = df_payments.agg(_sum("payment_value")).collect()[0][0]

df_gold_payment_breakdown = df_payments.groupBy("payment_type").agg(
    _round(_sum("payment_value"), 2).alias("valeur_totale"),
    count("*").alias("nb_transactions"),
    _round(avg("payment_value"), 2).alias("valeur_moyenne")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_payments) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_gold_payment_breakdown.show()

In [ ]:
# Qualité des données : transactions voucher à valeur nulle
zero_value_vouchers = df_payments.filter(
    (col("payment_type") == "voucher") & (col("payment_value") == 0)
).count()
print(f"Transactions voucher avec valeur zéro : {zero_value_vouchers}")

## Parcellement moyen par type de paiement

In [ ]:
df_installments_by_type = df_payments.groupBy("payment_type").agg(
    _round(avg("payment_installments"), 2).alias("parcelles_moyennes"),
    count("*").alias("nb_transactions")
).orderBy(col("nb_transactions").desc())

df_installments_by_type.show()

## Panier moyen par commande
Agrégation des paiements au niveau commande avant jointure.

In [ ]:
df_payments_agg = df_payments.groupBy("order_id").agg(
    _sum("payment_value").alias("valeur_totale_payee"),
    _max("payment_installments").alias("max_parcelles"),
    concat_ws(",", collect_set("payment_type")).alias("types_paiement")
)

In [ ]:
df_orders_for_payment_analysis = df_orders.filter(
    ~col("order_status").isin(["canceled", "unavailable"])
)

df_gold_payments = df_payments_agg.join(
    df_orders_for_payment_analysis.select("order_id", "customer_id", "order_status", "order_purchase_timestamp"),
    "order_id", "inner"
)

In [ ]:
df_gold_payments.agg(
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).show()

df_gold_payments.groupBy("types_paiement").agg(
    count("*").alias("nb_commandes"),
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).orderBy(col("nb_commandes").desc()).show()

## Paiements par état du client (demande géographique)

In [ ]:
total_customer_payments = df_payments_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "inner"
).agg(_sum("valeur_totale_payee")).collect()[0][0]

df_payments_by_customer_state = df_payments_agg.join(
    df_orders.select("order_id", "customer_id"), "order_id", "inner"
).join(
    df_customers.select("customer_id", "customer_state"), "customer_id", "inner"
).groupBy("customer_state").agg(
    _round(_sum("valeur_totale_payee"), 2).alias("valeur_totale"),
    count("*").alias("nb_commandes"),
    _round(avg("valeur_totale_payee"), 2).alias("panier_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_customer_payments) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_payments_by_customer_state.show(27)

## Paiements par région du client

In [ ]:
region_map = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

mapping_expr = create_map([lit(x) for x in chain(*region_map.items())])

df_payments_by_customer_region = df_payments_by_customer_state.withColumn(
    "region", mapping_expr[col("customer_state")]
).groupBy("region").agg(
    _round(_sum("valeur_totale"), 2).alias("valeur_totale"),
    _sum("nb_commandes").alias("nb_commandes")
).orderBy(col("valeur_totale").desc())

df_payments_by_customer_region.show()

## Validation — vendeurs par commande

In [ ]:
sellers_per_order = df_items.groupBy("order_id").agg(countDistinct("seller_id").alias("nb_sellers"))
sellers_per_order.groupBy("nb_sellers").count().orderBy("nb_sellers").show()

## Chiffre d'affaires par état du vendeur (offre géographique)

In [ ]:
total_seller_revenue = df_items.agg(_sum("price")).collect()[0][0]

df_revenue_by_seller_state = df_items.join(
    df_sellers.select("seller_id", "seller_state"), "seller_id", "inner"
).groupBy("seller_state").agg(
    _round(_sum("price"), 2).alias("valeur_totale"),
    count("*").alias("nb_items"),
    _round(avg("price"), 2).alias("prix_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_seller_revenue) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_revenue_by_seller_state.show(30)

## Top vendeurs par chiffre d'affaires généré
Basé sur le revenu reconnu (commandes livrées).

In [ ]:
df_top_sellers = df_items.join(
    df_orders_delivered.select("order_id"), "order_id", "inner"
).join(
    df_sellers.select("seller_id", "seller_city", "seller_state"), "seller_id", "inner"
).groupBy("seller_id", "seller_city", "seller_state").agg(
    _round(_sum("price"), 2).alias("ca_genere"),
    count("*").alias("nb_items_vendus"),
    _round(avg("price"), 2).alias("prix_moyen")
).orderBy(col("ca_genere").desc())

df_top_sellers.show(20, truncate=False)

## Chiffre d'affaires par catégorie de produit
`product_category_name` déjà nettoyé en silver (valeur 'unknown' si absente).

In [ ]:
total_category_revenue = df_items.agg(_sum("price")).collect()[0][0]

df_revenue_by_category = df_items.join(
    df_products.select("product_id", "product_category_name"), "product_id", "left"
).join(
    df_pcnt.select("product_category_name", "product_category_name_english"),
    "product_category_name", "left"
).withColumn(
    "categorie",
    coalesce(col("product_category_name_english"), col("product_category_name"))
).groupBy("categorie").agg(
    _round(_sum("price"), 2).alias("valeur_totale"),
    count("*").alias("nb_items"),
    _round(avg("price"), 2).alias("prix_moyen")
).withColumn(
    "pourcentage_valeur",
    _round((col("valeur_totale") / total_category_revenue) * 100, 2)
).orderBy(col("valeur_totale").desc())

df_revenue_by_category.show(30)

## Sauvegarde en zone gold

In [ ]:
df_gold_revenue_delivered.write.mode("overwrite").parquet("../data/gold/account/revenue_delivered/")
df_gold_revenue_in_transit.write.mode("overwrite").parquet("../data/gold/account/revenue_in_transit/")
df_gold_revenue_excluded.write.mode("overwrite").parquet("../data/gold/account/revenue_excluded/")
df_monthly_revenue.write.mode("overwrite").parquet("../data/gold/account/monthly_revenue/")
df_gold_payment_breakdown.write.mode("overwrite").parquet("../data/gold/account/payment_breakdown/")
df_installments_by_type.write.mode("overwrite").parquet("../data/gold/account/installments_by_type/")
df_payments_by_customer_state.write.mode("overwrite").parquet("../data/gold/account/payments_by_customer_state/")
df_payments_by_customer_region.write.mode("overwrite").parquet("../data/gold/account/payments_by_customer_region/")
df_revenue_by_seller_state.write.mode("overwrite").parquet("../data/gold/account/revenue_by_seller_state/")
df_top_sellers.write.mode("overwrite").parquet("../data/gold/account/top_sellers/")
df_revenue_by_category.write.mode("overwrite").parquet("../data/gold/account/revenue_by_category/")